# Fictional Context Story – Public Health NGO Dataset

### Context
In 2024, the international NGO **Global Health Reach (GHR)** launched a city-wide initiative to monitor and improve primary healthcare services across two hospitals and three clinics located in different neighborhoods of the fictional bilingual West-African city of **Kambara City**. The city is officially bilingual (French and English), but linguistic practices vary widely by neighborhood.

### The Clinics
Each of the five neighborhood centers operates semi-independently, using its own staff, tools, workflows, and data entry practices. This decentralization creates significant inconsistencies and quality issues across the datasets, which is why GHR recruited a skilled data analyst to lead the consolidation and cleaning effort.

### Francophone neighborhoods (3 clinics)
- **Clinique Saint-Bernard (CSB)** – located in a semi-urban area, with older computer systems and paper-based archives.
- **Centre Médical Luko (CML)** – situated in a densely populated neighborhood; nurses often enter data in French and occasionally mix in local dialect terms.
- **Hôpital de Kanza (HKZ)** – the largest francophone hospital; recently digitized, but migration caused inconsistent data formats.

### Anglophone neighborhoods (2 clinics)
- **Abeni Health Center (LAHC)** – modern EMR system in use, but frequent software updates cause CSV export inconsistencies.
- **West Karumo Medical Post (WKMP)** – located in a remote part of the city, sometimes offline, resulting in delayed or partially missing records. Data is first recorded manually and then typed by volunteers with varying accuracy.

### Longitudinal Data
Patients are tracked across multiple visits throughout the year, allowing GHR to monitor individual health trends over time. Each patient may have 1–5 visits, with measurements such as weight, height, temperature, blood pressure, diagnosis, and vaccination status recorded for each visit. Notes may include logistical observations or qualitative feedback about patient satisfaction.

### Objective
GHR’s program manager mandated a comprehensive data quality assessment to clean, reconcile, and standardize the information from all five facilities. The mission involves identifying inconsistencies, harmonizing formats and definitions, resolving missing or contradictory records, handling outliers, and producing a consolidated dataset robust enough to support evidence-based decision-making and donor-required reporting in English. This effort is intended to lay the groundwork for stronger data governance, reliable longitudinal health monitoring, and actionable insights across all neighborhoods of Kambara City.

### Scope of This Notebook (Part 1)

This notebook focuses exclusively on **Part 1 of the data cleaning pipeline**, which includes:  

- **Data loading and initial overview** of the five CSV files  
- **Standardization and renaming of raw columns** across clinics  
- **Administrative column cleaning and harmonization without performing any medical or numerical conversions at this stage**

**Author:** J-F Jutras  
**Date:** December 2025  
**Dataset:** GHR-Dataset (5 CSV files)

## 1.1-Data Loading and Overview

In [1]:
import pandas as pd
import os

#Folder where all CSV dataset files are located
folder = "/kaggle/input/ghr-dataset/"

#List only CSV files in the folder
files = [f for f in os.listdir(folder) if f.endswith(".csv")]

#Dictionary to store each dataset as a pandas DataFrame
dfs = {}

#Loop through each CSV file and load it
for file in files:
    path = os.path.join(folder, file)
    name = os.path.splitext(file)[0]  #Use filename (without extension) as the key

    try:
        #Initial read of the CSV file
        #'sep=None' and 'engine=python' allow pandas to auto-detect the separator
        #encoding='latin1' works for most files; errors='ignore' prevents crashes on bad lines
        df = pd.read_csv(path, sep = None, engine = 'python', encoding = 'latin1', on_bad_lines = 'skip')

        #Check for single-column issue (common when separators are inconsistent)
        if df.shape[1] == 1:
            #Detect separator: semicolon, comma, or tab
            sep = ';' if ';' in df.iloc[0,0] else (',' if ',' in df.iloc[0,0] else '\t')
            
            #Split the single column into multiple columns
            df = df.iloc[:,0].str.split(sep, expand=True)
            
            #Use the first row as header if it does not contain NaN
            df.columns = df.iloc[0]
            df = df[1:].reset_index(drop=True)

        #Store the cleaned DataFrame in the dictionary
        dfs[name] = df
        print(f"Loaded {name}, shape: {df.shape}")

    except Exception as e:
        #Print any errors encountered while loading this file
        print(f"Error loading {name}: {e}")

Loaded Donnees_Kanza_Kambara, shape: (100, 22)
Loaded Donnees_CSB_Kambara, shape: (221, 19)
Loaded Donnees_WKMP_Kambara, shape: (150, 19)
Loaded Donnees_Abeni_Kambara, shape: (422, 22)
Loaded Donnees_Luko_Kambara, shape: (204, 22)


| Nom de colonne (FR) | Column Name (EN) | Description | Scope |
|--------------------|-----------------|-------------|-------|
| id_patient | id_patient | Unique patient identifier | Clinic/Hospital |
| num_visite | visit_number | Visit number for the patient (1–5) | Clinic/Hospital |
| date_visite | visit_date | Date of the visit | Clinic/Hospital |
| nom_clinique | clinic_name | Name of the clinic or hospital where the visit occurred | Clinic/Hospital |
| langue_patient | patient_language | Patient's preferred language (French/English) | Clinic/Hospital |
| prenom | first_name | Patient's first name | Clinic/Hospital |
| nom | last_name | Patient's last name | Clinic/Hospital |
| sexe | sex | Patient's sex | Clinic/Hospital |
| date_naissance | birth_date | Patient's date of birth | Clinic/Hospital |
| age | age | Patient's age at the time of the visit | Clinic/Hospital |
| taille | height | Patient's height | Clinic/Hospital |
| poids | weight | Patient's weight | Clinic/Hospital |
| temperature | temperature | Body temperature measured during the visit | Clinic/Hospital |
| pression_sanguine | blood_pressure | Blood pressure measurement | Clinic/Hospital |
| pouls | pulse | Pulse rate (beats per minute) | Clinic/Hospital |
| saturation_oxygene | oxygen_saturation | Blood oxygen saturation | Clinic/Hospital |
| diagnostic | diagnosis | Diagnosis given during the visit. Possible values: Malaria, Tuberculosis, Meningitis, Trypanosomiasis, Cholera, Typhoid, Dengue | Clinic/Hospital |
| statut_vaccinal | vaccination_status | Vaccination status. Possible values: Complete, Partial, Not vaccinated | Clinic/Hospital |
| hospitalisation_necessaire | hospitalization_required | Whether hospitalization was required. Possible values: Y, N | Clinic/Hospital |
| jours_hospitalisation | hospitalization_days | Number of days hospitalized if required | Hospital only |
| analyses_sanguines_necessaires | blood_tests_required | Whether blood tests were required. Possible values: Y, N | Hospital only |
| imagerie_necessaire | imaging_required | Whether imaging exams (X-ray, ultrasound, etc.) were required. Possible values: Y, N | Hospital only |


In [2]:
import warnings
#Ignore RuntimeWarnings from pandas formatting when printing DataFrames.
#These warnings are usually caused by NaN or mixed-type columns being interpreted as numeric.
warnings.simplefilter(action='ignore', category=RuntimeWarning)

#Inspect basic information for each dataset
for name, df in dfs.items():
    print(f"\nDataset: {name}")
    print("Columns:", df.columns.tolist())
    print(df.head())


Dataset: Donnees_Kanza_Kambara
Columns: ['id_patient', 'num_visite', 'date_visite', 'nom_clinique', 'langue_patient', 'prenom', 'nom', 'sexe', 'date_naissance', 'age', 'taille', 'poids', 'temperature', 'pression_sanguine', 'pouls', 'saturation_oxygene', 'diagnostic', 'statut_vaccinal', 'hospitalisation_necessaire', 'jours_hospitalisation', 'analyses_sanguines_necessaires', 'imagerie_necessaire']
  id_patient  num_visite date_visite      nom_clinique langue_patient  \
0       H101           1  2024-03-01  Hôpital de Kanza       Français   
1       H101           2  10/03/2024  Hôpital de Kanza       français   
2       H102           1  05/03/2024                HK       Francais   
3       H102           2  20-03-2024                HK       Français   
4       H103           1  15/03/2024  Hôpital de Kanza       français   

    prenom     nom sexe date_naissance  age  ... temperature  \
0  Aminata  DIALLO    F     1990-01-01   34  ...      39.5 C   
1  aminata  diallo    f     01/01

## 1.2-Standardize and Rename Columns

In [3]:
import numpy as np

#Before starting, create copies of all original DF
dfs_original = {name: df.copy(deep=True) for name, df in dfs.items()}

#Mapping of all possible column names → standard English names
Column_mapping = {
    #Identifiers and visits
    'id_patient': 'id_patient', 'patient_id': 'id_patient',
    'num_visite': 'visit_number', 'visit_num': 'visit_number',
    'date_visite': 'visit_date', 'visit_date': 'visit_date',
    'nom_clinique': 'clinic_name', 'clinic_name': 'clinic_name',
    'langue_patient': 'patient_language', 'patient_language': 'patient_language',
    #Patient info
    'prenom': 'first_name', 'first_name': 'first_name',
    'nom': 'last_name', 'last_name': 'last_name',
    'sexe': 'sex', 'gender': 'sex',
    'date_naissance': 'birth_date', 'date_of_birth': 'birth_date',
    'age': 'age',
    #Measurements
    'taille': 'height', 'height': 'height',
    'poids': 'weight', 'weight': 'weight',
    'temperature': 'temperature',
    'pression_sanguine': 'blood_pressure', 'blood_pressure': 'blood_pressure',
    'pouls': 'pulse', 'pulse': 'pulse',
    'saturation_oxygene': 'oxygen_saturation', 'oxygen_saturation': 'oxygen_saturation',
    #Diagnosis and vaccination
    'diagnostic': 'diagnosis',
    'statut_vaccinal': 'vaccination_status',
    #Hospitalization
    'hospitalisation_necessaire': 'hospitalization_required',
    'jours_hospitalisation': 'hospitalization_days',
    #Tests and imaging
    'analyses_sanguines_necessaires': 'blood_tests_required',
    'imagerie_necessaire': 'imaging_required'
}

#List of all standard columns that every dataset should have
Standard_columns = [
    'id_patient', 'visit_number', 'visit_date', 'clinic_name', 'patient_language',
    'first_name', 'last_name', 'sex', 'birth_date', 'age', 'height', 'weight', 
    'temperature', 'blood_pressure', 'pulse', 'oxygen_saturation', 'diagnosis',
    'vaccination_status', 'hospitalization_required', 'hospitalization_days',
    'blood_tests_required', 'imaging_required'
]

#Function to standardize column names and add missing columns
def standardize_columns(df):
    #Rename columns according to mapping
    df = df.rename(columns = lambda x: Column_mapping.get(x, x))
    
    #Add missing columns with NaN
    for col in Standard_columns:
        if col not in df.columns:
            df[col] = np.nan
    
    #Reorder columns to match standard
    df = df[Standard_columns]
    
    return df

#Apply function to all datasets
for name in dfs:
    dfs[name] = standardize_columns(dfs[name])
    print(f"{name} standardized. Shape: {dfs[name].shape}")

Donnees_Kanza_Kambara standardized. Shape: (100, 22)
Donnees_CSB_Kambara standardized. Shape: (221, 22)
Donnees_WKMP_Kambara standardized. Shape: (150, 22)
Donnees_Abeni_Kambara standardized. Shape: (422, 22)
Donnees_Luko_Kambara standardized. Shape: (204, 22)


## 1.3-Raw Column Cleaning and Harmonization - Administrative Columns

Before standardizing data types across all datasets, we perform an initial cleaning of the raw columns. This step ensures that values are consistent, extraneous characters are removed, and textual formatting (e.g., capitalization, whitespace) is normalized. 

**Why this step is important:**  
- Prevents loss of information when converting columns to numeric or datetime types.  
- Reduces errors caused by inconsistent formatting across different clinics and hospitals.  
- Prepares categorical data for reliable type conversion.  
- Harmonizes variations in text for later analysis.  

This approach allows us to safely apply type standardization while minimizing unintended data corruption.


In [4]:
#Define administrative columns
admin_cols = ['id_patient', 'visit_number', 'visit_date', 'clinic_name', 'patient_language']

#Loop through datasets and inspect unique values for administrative columns
for name, df in dfs.items():
    print(f"\nDataset: {name}")
    for col in admin_cols:
        if col in df.columns:
            unique_vals = df[col].dropna().unique()
            print(f"Column: {col} (unique values count : {len(unique_vals)})")


Dataset: Donnees_Kanza_Kambara
Column: id_patient (unique values count : 50)
Column: visit_number (unique values count : 2)
Column: visit_date (unique values count : 100)
Column: clinic_name (unique values count : 2)
Column: patient_language (unique values count : 3)

Dataset: Donnees_CSB_Kambara
Column: id_patient (unique values count : 100)
Column: visit_number (unique values count : 5)
Column: visit_date (unique values count : 97)
Column: clinic_name (unique values count : 4)
Column: patient_language (unique values count : 7)

Dataset: Donnees_WKMP_Kambara
Column: id_patient (unique values count : 40)
Column: visit_number (unique values count : 6)
Column: visit_date (unique values count : 81)
Column: clinic_name (unique values count : 3)
Column: patient_language (unique values count : 4)

Dataset: Donnees_Abeni_Kambara
Column: id_patient (unique values count : 306)
Column: visit_number (unique values count : 5)
Column: visit_date (unique values count : 199)
Column: clinic_name (uni

In [5]:
#Inspect unique values for clinic_name and patient_language
for name, df in dfs.items():
    print(f"\nDataset: {name}")
    if 'clinic_name' in df.columns:
        print("Clinic names:", df['clinic_name'].unique())
    if 'patient_language' in df.columns:
        print("Patient languages:", df['patient_language'].unique())


Dataset: Donnees_Kanza_Kambara
Clinic names: ['Hôpital de Kanza' 'HK']
Patient languages: ['Français' 'français' 'Francais']

Dataset: Donnees_CSB_Kambara
Clinic names: ['Saint-Bernard Clinic' 'CSB' 'Saint-Bernard Clinique' 'Saint-Bernard']
Patient languages: ['Francais' 'français' 'Français' 'FRANCAIS' 'anglais' 'Anglais'
 'francais']

Dataset: Donnees_WKMP_Kambara
Clinic names: ['West Karumo Medical Post' 'WKMP' 'West Karumo Med Post']
Patient languages: ['English' 'english' 'ENGLISH' 'French']

Dataset: Donnees_Abeni_Kambara
Clinic names: ['Lake Abeni Health Center' 'Lake Abeni Health Ctr' None
 'Lake Abeni Health Cntr' 'West Karumo Medical Post']
Patient languages: ['English' None 'French' 'Englihs']

Dataset: Donnees_Luko_Kambara
Clinic names: ['Centre Médical de Luko' 'CML' 'Hôpital de Kanza']
Patient languages: ['Français' 'français' 'Francais']


In [6]:
import numpy as np

#Mapping for clinic names
clinic_name_map = {
    'Hôpital de Kanza': 'Hôpital de Kanza',
    'HK': 'Hôpital de Kanza',
    'Saint-Bernard Clinic': 'Clinique Saint-Bernard',
    'CSB': 'Clinique Saint-Bernard',
    'Saint-Bernard Clinique': 'Clinique Saint-Bernard',
    'Saint-Bernard': 'Clinique Saint-Bernard',
    'West Karumo Medical Post': 'West Karumo Medical Post',
    'WKMP': 'West Karumo Medical Post',
    'West Karumo Med Post': 'West Karumo Medical Post',
    'Lake Abeni Health Center': 'Lake Abeni Health Center',
    'Lake Abeni Health Ctr': 'Lake Abeni Health Center',
    'Lake Abeni Health Cntr': 'Lake Abeni Health Center',
    'Centre Médical de Luko': 'Centre Médical de Luko',
    'CML': 'Centre Médical de Luko'
}

#Mapping for patient languages
language_map = {
    'Français': 'French',
    'français': 'French',
    'Francais': 'French',
    'francais': 'French',
    'FRANCAIS': 'French',
    'anglais': 'English',
    'Anglais': 'English',
    'English': 'English',
    'english': 'English',
    'ENGLISH': 'English',
    'Englihs': 'English'
}

#Apply mappings to all datasets
for name, df in dfs.items():
    if 'clinic_name' in df.columns:
        df['clinic_name'] = df['clinic_name'].map(clinic_name_map).fillna(df['clinic_name'])
    if 'patient_language' in df.columns:
        df['patient_language'] = df['patient_language'].map(language_map).fillna(df['patient_language'])

#Quick check
for name, df in dfs.items():
    print(f"\nDataset: {name}")
    if 'clinic_name' in df.columns:
        print("Clinic names:", df['clinic_name'].unique())
    if 'patient_language' in df.columns:
        print("Patient languages:", df['patient_language'].unique())


Dataset: Donnees_Kanza_Kambara
Clinic names: ['Hôpital de Kanza']
Patient languages: ['French']

Dataset: Donnees_CSB_Kambara
Clinic names: ['Clinique Saint-Bernard']
Patient languages: ['French' 'English']

Dataset: Donnees_WKMP_Kambara
Clinic names: ['West Karumo Medical Post']
Patient languages: ['English' 'French']

Dataset: Donnees_Abeni_Kambara
Clinic names: ['Lake Abeni Health Center' None 'West Karumo Medical Post']
Patient languages: ['English' None 'French']

Dataset: Donnees_Luko_Kambara
Clinic names: ['Centre Médical de Luko' 'Hôpital de Kanza']
Patient languages: ['French']


Some datasets contain more than one clinic per patient, which may reflect transfers or visits to different centers. We will now correct these records to ensure consistency, while keeping the idea of multiple clinic visits in mind for the rest of the analysis.

In [7]:
#Correct West Karumo in Abeni Dataset
dfs['Donnees_Abeni_Kambara']['clinic_name'] = dfs['Donnees_Abeni_Kambara']['clinic_name'].replace({
    'West Karumo Medical Post': 'Lake Abeni Health Center'
})

#Correct Kanza in Luko
dfs['Donnees_Luko_Kambara']['clinic_name'] = dfs['Donnees_Luko_Kambara']['clinic_name'].replace({
    'Hôpital de Kanza' : 'Centre Médical de Luko'
})

In the Abeni dataset, we also observe None values in clinic_name and patient_language. This appears to be the only dataset with such missing values, likely due to issues during column parsing or inconsistent data entry.

In [8]:
#Select Abeni dataset
df_abeni = dfs['Donnees_Abeni_Kambara']

#Rows where patient_language is NaN
missing_language = df_abeni[df_abeni['patient_language'].isna()]
print(missing_language[['id_patient', 'patient_language']])

#Rows where clinic_name is NaN
clinic_name = df_abeni[df_abeni['clinic_name'].isna()]
print(clinic_name[['id_patient', 'clinic_name']])

                                            id_patient patient_language
2    1001,3,10/10/2024,Lake Abeni Health Center,Eng...             None
5    1003,1,2024-08-20,Lake Abeni Health Center,Fre...             None
9    1004,3,2024-08-01,Lake Abeni Health Center,Eng...             None
15   1008,1,2024-02-14,Lake Abeni Health Center,Eng...             None
22   1013,1,2024-09-05,Lake Abeni Health Ctr,Englis...             None
27   1016,1,2024-05-01,Lake Abeni Health Center,Eng...             None
35   1020,3,2024-11-15,Lake Abeni Health Center,Eng...             None
42   1025,1,2024-07-20,Lake Abeni Health Center,Eng...             None
55   1036,1,2024-07-01,Lake Abeni Health Center,Eng...             None
60   1041,1,2024-12-10,Lake Abeni Health Center,Eng...             None
66   1047,1,2024-06-25,Lake Abeni Health Center,Eng...             None
75   1056,1,2024-03-18,Lake Abeni Health Center,Eng...             None
93   1072,1,2024-08-08,Lake Abeni Health Center,Eng...          

In [9]:
import warnings

#Suppress SettingWithCopyWarning and FutureWarning temporarily
warnings.simplefilter(action='ignore', category=FutureWarning)
warnings.simplefilter(action='ignore', category=pd.errors.SettingWithCopyWarning)

#Reference to Abeni dataset
df_abeni = dfs['Donnees_Abeni_Kambara']

#Identify rows where 'id_patient' contains multiple fields (collapsed data)
#These rows need to be fully split into all standard columns
problematic_rows = df_abeni['id_patient'].str.contains(',', na = False)

#Split the collapsed rows into separate columns
split_cols = [
    'id_patient', 'visit_number', 'visit_date', 'clinic_name', 'patient_language',
    'first_name', 'last_name', 'sex', 'birth_date', 'age', 'height', 'weight',
    'temperature', 'blood_pressure', 'pulse', 'oxygen_saturation', 'diagnosis',
    'vaccination_status', 'hospitalization_required', 'hospitalization_days',
    'blood_tests_required', 'imaging_required'
]

#Split the collapsed strings
split_df = df_abeni.loc[problematic_rows, 'id_patient'].str.split(',', expand = True)

#Some rows might have fewer columns; ensure we have all 22 columns
for i, col in enumerate(split_cols):
    if i >= split_df.shape[1]:
        split_df[i] = np.nan

split_df = split_df.iloc[:, :len(split_cols)]
split_df.columns = split_cols

#Replace the problematic rows with the fully split DataFrame
df_abeni.loc[problematic_rows, split_cols] = split_df

#Standardize clinic names
clinic_map = {
    'Lake Abeni Health Center': 'Lake Abeni Health Center',
    'Lake Abeni Health Ctr': 'Lake Abeni Health Center',
    'Lake Abeni Health Cntr': 'Lake Abeni Health Center',
    'West Karumo Medical Post': 'Lake Abeni Health Center'
}
df_abeni['clinic_name'] = df_abeni['clinic_name'].map(clinic_map).fillna(df_abeni['clinic_name'])

#Standardize patient languages
language_map = {
    'English': 'English',
    'Englihs': 'English',
    'French': 'French',
    'Fre': 'French'
}
df_abeni['patient_language'] = df_abeni['patient_language'].map(language_map).fillna(df_abeni['patient_language'])

#Update the dfs dictionary
dfs['Donnees_Abeni_Kambara'] = df_abeni

#Quick verification
print(df_abeni.loc[problematic_rows, split_cols])
print("Unique clinic names:", df_abeni['clinic_name'].unique())
print("Unique patient languages:", df_abeni['patient_language'].unique())


    id_patient visit_number  visit_date               clinic_name  \
2         1001            3  10/10/2024  Lake Abeni Health Center   
5         1003            1  2024-08-20  Lake Abeni Health Center   
9         1004            3  2024-08-01  Lake Abeni Health Center   
15        1008            1  2024-02-14  Lake Abeni Health Center   
22        1013            1  2024-09-05  Lake Abeni Health Center   
27        1016            1  2024-05-01  Lake Abeni Health Center   
35        1020            3  2024-11-15  Lake Abeni Health Center   
42        1025            1  2024-07-20  Lake Abeni Health Center   
55        1036            1  2024-07-01  Lake Abeni Health Center   
60        1041            1  2024-12-10  Lake Abeni Health Center   
66        1047            1  2024-06-25  Lake Abeni Health Center   
75        1056            1  2024-03-18  Lake Abeni Health Center   
93        1072            1  2024-08-08  Lake Abeni Health Center   
104       1083            1  2024-

In [10]:
import re
import numpy as np

#We are only investigating existing date patterns (NOT cleaning yet)
#This helps us discover how dates are actually written in all datasets.

date_cols = ['visit_date']

def extract_pattern(date_str):
    pattern = ""
    for c in str(date_str):
        if c.isdigit():
            pattern += "d"
        elif c.isalpha():
            pattern += "a"
        else:
            pattern += c
    return pattern

for name, df in dfs.items():
    print(f"\n=== Dataset: {name} ===")

    for col in date_cols:
        if col not in df.columns:
            continue
    
        print(f"\nColumn: {col}")

        #Unique non-null values
        unique_dates = df[col].dropna().astype(str).unique()

        #Extract structural patterns
        patterns_found = set()
        for val in unique_dates:
            pattern = extract_pattern(val)
            patterns_found.add(pattern)

        #Display discovered patterns only
        print("Detected date patterns:")
        for pattern in patterns_found:
            print(f"  {pattern}")


=== Dataset: Donnees_Kanza_Kambara ===

Column: visit_date
Detected date patterns:
  dd-dd-dddd
  dd/dd/dddd
  dddd-dd-dd

=== Dataset: Donnees_CSB_Kambara ===

Column: visit_date
Detected date patterns:
  dddd/dd/dd
  dd-dd-dddd
  dd/dd/dd
  dd/dd/dddd
  dddd-dd-dd

=== Dataset: Donnees_WKMP_Kambara ===

Column: visit_date
Detected date patterns:
  dddd/dd/dd
  dd-dd-dddd
  dd/dd/dd
  dd/dd/dddd
  dddd-dd-dd

=== Dataset: Donnees_Abeni_Kambara ===

Column: visit_date
Detected date patterns:
  dddd/dd/dd
  dd-dd-dddd
  dd.dd.dddd
  dd/dd/dddd
  dddd-dd-dd

=== Dataset: Donnees_Luko_Kambara ===

Column: visit_date
Detected date patterns:
  dd-dd-dddd
  dd/dd/dddd
  dddd-dd-dd


In [11]:
#Create a function to standardize dates
def clean_dates(series):
    #Convert to string and remove trailing spaces
    cleaned = series.astype(str).str.strip()

    #Normalize separators to "-"
    #This allows easier pattern matching in later steps
    cleaned = cleaned.str.replace(r"[./]", "-", regex = True)

    #Convert 2-digit years to 4-digit years (e.g., "05-03-24" → "05-03-2024")
    def fix_two_digit_year(val):
        #Pattern: DD-MM-YY
        if re.match(r"^\d{1,2}-\d{1,2}-\d{2}$", val):
            d, m, y = val.split("-")
            return f"{d}-{m}-20{y}"
        return val

    cleaned = cleaned.apply(fix_two_digit_year)

    #Reorder DD-MM-YYYY to YYYY-MM-DD (the canonical ISO format)
    def reorder_if_needed(val):
        #Pattern: DD-MM-YYYY → flip to YYYY-MM-DD
        if re.match(r"^\d{1,2}-\d{1,2}-\d{4}$", val):
            d, m, y = val.split("-")
            return f"{y}-{m}-{d}"

        #Already correct: YYYY-MM-DD
        if re.match(r"^\d{4}-\d{1,2}-\d{1,2}$", val):
            return val

        #Anything else will be handled by pandas later
        return val

    cleaned = cleaned.apply(reorder_if_needed)

    return cleaned

#Apply cleaning to all DF in the dictionary
for name, df in dfs.items():
    if "visit_date" in df.columns:
        df["visit_date"] = clean_dates(df["visit_date"])

In [12]:
#Clean remaining administrative columns and convert to string
for name, df in dfs.items():
    for col in admin_cols:
        if col not in df.columns:
            continue

        #Convert raw values to string + strip whitespace
        df[col] = df[col].astype(str).str.strip()

        #Normalize missing-value markers → NaN
        df[col] = df[col].replace({
            "": np.nan,
            " ": np.nan,
            "\t": np.nan,
            "none": np.nan,
            "None": np.nan,
            "NULL": np.nan,
            "null": np.nan,
            "NaN": np.nan,
            "nan": np.nan
        })

        #Keep everything as string for staging
        df[col] = df[col].astype("string")

#Quick check
for name, df in dfs.items():
    print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 22 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   id_patient                100 non-null    string 
 1   visit_number              100 non-null    string 
 2   visit_date                100 non-null    string 
 3   clinic_name               100 non-null    string 
 4   patient_language          100 non-null    string 
 5   first_name                100 non-null    object 
 6   last_name                 99 non-null     object 
 7   sex                       100 non-null    object 
 8   birth_date                100 non-null    object 
 9   age                       100 non-null    int64  
 10  height                    100 non-null    object 
 11  weight                    100 non-null    object 
 12  temperature               100 non-null    object 
 13  blood_pressure            100 non-null    object 
 14  pulse      

## 1.4-Summary

In this notebook, we completed **Part 1 of the data cleaning pipeline**, which included:  

- Loading the five CSV datasets from all clinics and hospitals.  
- Performing an initial overview to inspect data structure, missing values, and inconsistencies.  
- Standardizing and renaming columns to ensure consistent naming across all sources.  
- Cleaning and harmonizing administrative columns (e.g., patient IDs, visit dates) without converting medical or numerical values.


## 1.5-Save Datasets

In [13]:
import os

#Folder to save cleaned CSVs
save_folder = "/kaggle/working/cleaned_csvs"
os.makedirs(save_folder, exist_ok = True)

#Save each DataFrame in dfs
for name, df in dfs.items():
    file_path = os.path.join(save_folder, f"{name}_cleaned.csv")
    df.to_csv(file_path, index = False)
    print(f"Saved: {file_path}")

Saved: /kaggle/working/cleaned_csvs/Donnees_Kanza_Kambara_cleaned.csv
Saved: /kaggle/working/cleaned_csvs/Donnees_CSB_Kambara_cleaned.csv
Saved: /kaggle/working/cleaned_csvs/Donnees_WKMP_Kambara_cleaned.csv
Saved: /kaggle/working/cleaned_csvs/Donnees_Abeni_Kambara_cleaned.csv
Saved: /kaggle/working/cleaned_csvs/Donnees_Luko_Kambara_cleaned.csv
